# Построение дневной витрины признаков



## Задание

#### Контекст

Перед обучением модели нужно из событийной истории построить витрину признаков без утечки в будущее.

#### Даны файлы

`actions_log.csv`

* `customer_id`
* `action_dttm`
* `action_type` - `call`, `sms`, `promise`, `payment`
* `action_amount` - сумма действия, для `payment` и `promise`, иначе `0`

`daily_snapshot.csv`

* `customer_id`
* `snapshot_date`
* `dpd`
* `outstanding_amount`

#### Что нужно сделать

Для каждой строки из `daily_snapshot.csv` постройте признаки, используя только события строго раньше `snapshot_date`:

* `calls_last_7d`
* `sms_last_7d`
* `promises_last_30d`
* `payments_amount_last_30d`
* `days_since_last_payment`
* `promised_amount_last_30d`

После этого:

* соберите итоговую витрину в один `DataFrame`;
* найдите топ-20 клиентов с максимальным числом контактов (`calls_last_7d + sms_last_7d`) и нулевой суммой оплат за последние 30 дней.

#### Важное условие

Нельзя использовать события на `snapshot_date` и позже.

#### Что прислать

Код, итоговую схему витрины и пример выборки топ-20 клиентов.

## Решение

### Импорт и загрузка

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
actions = pd.read_csv('actions_log.csv')
snapshot = pd.read_csv('daily_snapshot.csv')

### Quick Look

In [ ]:
actions.head()

,customer_id,action_dttm,action_type,action_amount
0,400000,2026-01-01 14:59:00,payment,13309.31
1,400000,2026-01-01 19:11:00,call,0.00
2,400000,2026-01-08 14:37:00,payment,6772.92
3,400000,2026-01-10 08:04:00,call,0.00
4,400000,2026-01-12 14:06:00,payment,8731.15


In [ ]:
snapshot.head()

,customer_id,snapshot_date,dpd,outstanding_amount
0,400000,2026-03-11,10,19000.14
1,400000,2026-03-15,22,16914.99
2,400001,2026-03-30,26,16142.60
3,400001,2026-04-03,33,16326.84
4,400002,2026-02-21,90,44636.91


In [ ]:
actions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14954 entries, 0 to 14953
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_id    14954 non-null  int64  
 1   action_dttm    14954 non-null  object 
 2   action_type    14954 non-null  object 
 3   action_amount  14954 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 467.4+ KB


In [ ]:
snapshot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         4500 non-null   int64  
 1   snapshot_date       4500 non-null   object 
 2   dpd                 4500 non-null   int64  
 3   outstanding_amount  4500 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 140.8+ KB


In [ ]:
actions['action_type'].value_counts(dropna=False)

,count
action_type,
call,6220
sms,4341
payment,2284
promise,2109


In [ ]:
snapshot['customer_id'].nunique(), len(snapshot)

(1500, 4500)

### Приведение дат

In [ ]:
actions['action_dttm'] = pd.to_datetime(actions['action_dttm'])
snapshot['snapshot_date'] = pd.to_datetime(snapshot['snapshot_date'])

In [ ]:
actions.info()
print('-'*50)
snapshot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14954 entries, 0 to 14953
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customer_id    14954 non-null  int64         
 1   action_dttm    14954 non-null  datetime64[ns]
 2   action_type    14954 non-null  object        
 3   action_amount  14954 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 467.4+ KB
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_id         4500 non-null   int64         
 1   snapshot_date       4500 non-null   datetime64[ns]
 2   dpd                 4500 non-null   int64         
 3   outstanding_amount  4500 non-null   float64       
dtypes: da

### Merge snapshot + actions

In [ ]:
snapshot_actions = snapshot.merge(actions,
    on='customer_id',
    how='left')

In [ ]:
snapshot_actions.head()

,customer_id,snapshot_date,dpd,outstanding_amount,action_dttm,action_type,action_amount
0,400000,2026-03-11,10,19000.14,2026-01-01 14:59:00,payment,13309.31
1,400000,2026-03-11,10,19000.14,2026-01-01 19:11:00,call,0.00
2,400000,2026-03-11,10,19000.14,2026-01-08 14:37:00,payment,6772.92
3,400000,2026-03-11,10,19000.14,2026-01-10 08:04:00,call,0.00
4,400000,2026-03-11,10,19000.14,2026-01-12 14:06:00,payment,8731.15


In [ ]:
snapshot_actions.shape

(44972, 7)

### Фильтрация по датам

In [ ]:
snapshot_actions = snapshot_actions[snapshot_actions['action_dttm'] < snapshot_actions['snapshot_date']]

In [ ]:
(snapshot_actions['action_dttm'] >= snapshot_actions['snapshot_date']).sum()

np.int64(0)

Остаются только события с action_dttm < snapshot_date, что предотвращает утечку информации из будущего (data leakage).

### Расчёт признаков по окнам 7 и 30 дней

In [ ]:
window_calls_7d = snapshot_actions[(snapshot_actions['action_type'] == 'call') & (snapshot_actions['action_dttm'] >= snapshot_actions['snapshot_date'] - pd.Timedelta(days=7))]

calls_last_7d = (window_calls_7d
    .groupby(['customer_id', 'snapshot_date'], as_index=False)
    .agg(calls_last_7d=('action_type', 'count')))

In [ ]:
window_sms_7d = snapshot_actions[(snapshot_actions['action_type'] == 'sms') & (snapshot_actions['action_dttm'] >= snapshot_actions['snapshot_date'] - pd.Timedelta(days=7))]

sms_last_7d = (window_sms_7d
    .groupby(['customer_id', 'snapshot_date'], as_index=False)
    .agg(sms_last_7d=('action_type', 'count')))

In [ ]:
window_promises_30d = snapshot_actions[(snapshot_actions['action_type'] == 'promise') & (snapshot_actions['action_dttm'] >= snapshot_actions['snapshot_date'] - pd.Timedelta(days=30))]

promises_agg = (window_promises_30d
    .groupby(['customer_id', 'snapshot_date'], as_index=False)
    .agg(promises_last_30d=('action_type', 'count'),
         promised_amount_last_30d=('action_amount', 'sum')))

In [ ]:
window_payments_30d = snapshot_actions[(snapshot_actions['action_type'] == 'payment') & (snapshot_actions['action_dttm'] >= snapshot_actions['snapshot_date'] - pd.Timedelta(days=30))]

payments_amount_last_30d = (window_payments_30d
    .groupby(['customer_id', 'snapshot_date'], as_index=False)
    .agg(payments_amount_last_30d=('action_amount', 'sum')))

In [ ]:
payments_all = snapshot_actions[snapshot_actions['action_type'] == 'payment']

days_since_last_payment = (payments_all
    .groupby(['customer_id', 'snapshot_date'], as_index=False)
    .agg(last_payment_date=('action_dttm', 'max')))

days_since_last_payment['days_since_last_payment'] = (days_since_last_payment['snapshot_date'] - days_since_last_payment['last_payment_date']).dt.days
days_since_last_payment = days_since_last_payment.drop(columns='last_payment_date')

### Итоговая витрина

In [ ]:
feature_mart = snapshot.copy()

feature_mart = feature_mart.merge(calls_last_7d, on=['customer_id', 'snapshot_date'], how='left')
feature_mart = feature_mart.merge(sms_last_7d, on=['customer_id', 'snapshot_date'], how='left')
feature_mart = feature_mart.merge(promises_agg, on=['customer_id', 'snapshot_date'], how='left')
feature_mart = feature_mart.merge(payments_amount_last_30d, on=['customer_id', 'snapshot_date'], how='left')
feature_mart = feature_mart.merge(days_since_last_payment[['customer_id', 'snapshot_date', 'days_since_last_payment']], on=['customer_id', 'snapshot_date'], how='left')

count_sum_cols = ['calls_last_7d',
                  'sms_last_7d',
                  'promises_last_30d',
                  'payments_amount_last_30d',
                  'promised_amount_last_30d']

feature_mart[count_sum_cols] = feature_mart[count_sum_cols].fillna(0)

In [ ]:
feature_mart.head()

,customer_id,snapshot_date,dpd,outstanding_amount,calls_last_7d,sms_last_7d,promises_last_30d,promised_amount_last_30d,payments_amount_last_30d,days_since_last_payment,contacts_last_7d
0,400000,2026-03-11,10,19000.14,0.0,0.0,0.0,0.00,0.00,57.0,0.0
1,400000,2026-03-15,22,16914.99,0.0,0.0,0.0,0.00,0.00,61.0,0.0
2,400001,2026-03-30,26,16142.60,0.0,1.0,1.0,10794.71,21487.22,0.0,1.0
3,400001,2026-04-03,33,16326.84,0.0,0.0,1.0,10794.71,21487.22,4.0,0.0
4,400002,2026-02-21,90,44636.91,1.0,0.0,0.0,0.00,0.00,40.0,1.0


In [ ]:
feature_mart.tail()

,customer_id,snapshot_date,dpd,outstanding_amount,calls_last_7d,sms_last_7d,promises_last_30d,promised_amount_last_30d,payments_amount_last_30d,days_since_last_payment,contacts_last_7d
4495,401498,2026-03-08,13,15018.98,0.0,0.0,0.0,0.00,0.00,NaN,0.0
4496,401498,2026-04-06,25,13529.55,1.0,0.0,1.0,10304.35,11592.19,22.0,1.0
4497,401499,2026-02-19,48,48450.98,1.0,0.0,0.0,0.00,6316.16,0.0,1.0
4498,401499,2026-02-24,53,44179.69,0.0,1.0,0.0,0.00,6316.16,5.0,1.0
4499,401499,2026-04-07,53,41331.81,0.0,1.0,0.0,0.00,0.00,47.0,1.0


In [ ]:
feature_mart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   customer_id               4500 non-null   int64         
 1   snapshot_date             4500 non-null   datetime64[ns]
 2   dpd                       4500 non-null   int64         
 3   outstanding_amount        4500 non-null   float64       
 4   calls_last_7d             4500 non-null   float64       
 5   sms_last_7d               4500 non-null   float64       
 6   promises_last_30d         4500 non-null   float64       
 7   promised_amount_last_30d  4500 non-null   float64       
 8   payments_amount_last_30d  4500 non-null   float64       
 9   days_since_last_payment   3061 non-null   float64       
 10  contacts_last_7d          4500 non-null   float64       
dtypes: datetime64[ns](1), float64(8), int64(2)
memory usage: 386.8 KB


### Итоговая схема витрины

| column_name | description |
|-------------|-------------|
| customer_id | идентификатор клиента |
| snapshot_date | дата среза |
| dpd | просрочка на дату среза |
| outstanding_amount | сумма задолженности на дату среза |
| calls_last_7d | количество звонков за последние 7 дней до snapshot_date |
| sms_last_7d | количество SMS за последние 7 дней до snapshot_date |
| promises_last_30d | количество обещаний за последние 30 дней до snapshot_date |
| payments_amount_last_30d | сумма платежей за последние 30 дней до snapshot_date |
| days_since_last_payment | число дней с момента последнего платежа до snapshot_date |
| promised_amount_last_30d | сумма обещаний за последние 30 дней до snapshot_date |

- Для count/sum-признаков отсутствие событий интерпретируется как 0.
- Для days_since_last_payment отсутствие исторических платежей оставляется как NaN.

### Top-20 клиентов

In [ ]:
feature_mart['contacts_last_7d'] = (feature_mart['calls_last_7d'] + feature_mart['sms_last_7d'])

top_clients_base = feature_mart[(feature_mart['payments_amount_last_30d'] == 0) & (feature_mart['contacts_last_7d'] > 0)].copy()

idx = top_clients_base.groupby('customer_id')['contacts_last_7d'].idxmax()

top_20_clients = (top_clients_base.loc[idx,['customer_id',
                                            'snapshot_date',
                                            'dpd',
                                            'outstanding_amount',
                                            'calls_last_7d',
                                            'sms_last_7d',
                                            'contacts_last_7d',
                                            'payments_amount_last_30d']]
                  .sort_values('contacts_last_7d', ascending=False).head(20).reset_index(drop=True))

In [ ]:
top_20_clients

,customer_id,snapshot_date,dpd,outstanding_amount,calls_last_7d,sms_last_7d,contacts_last_7d,payments_amount_last_30d
0,400784,2026-02-02,72,15362.99,1.0,4.0,5.0,0.0
1,400470,2026-02-14,51,31271.78,1.0,3.0,4.0,0.0
2,401296,2026-04-24,103,30408.20,2.0,2.0,4.0,0.0
3,400820,2026-04-09,85,35028.88,1.0,3.0,4.0,0.0
4,401164,2026-04-09,41,49944.68,2.0,2.0,4.0,0.0
5,400729,2026-04-12,114,34103.46,2.0,2.0,4.0,0.0
6,401486,2026-03-23,67,10598.69,3.0,1.0,4.0,0.0
7,400985,2026-02-09,33,78428.12,4.0,0.0,4.0,0.0
8,401423,2026-02-19,59,27350.83,2.0,1.0,3.0,0.0
9,400708,2026-02-17,50,49058.48,2.0,1.0,3.0,0.0
